In [1]:
import boto3
import os
from project_paths import PATHS, s3_uri


In [2]:
# Initialize SageMaker client
sagemaker_client = boto3.client('sagemaker', region_name='us-east-1')

In [3]:
# Configuration
model_name = os.getenv('SP_MODEL_NAME', 'oejr-prob-xgb-model-20260226')
print(f"Model Name: {model_name}")

endpoint_config_name = os.getenv('SP_ENDPOINT_CONFIG_NAME', 'oejr-prob-xgb-endpoint-config-20260226')
print(f"Endpoint Config Name: {endpoint_config_name}")

endpoint_name = os.getenv('SP_ENDPOINT_NAME', 'oejr-prob-xgb-endpoint-20260226')
print(f"Endpoint Name: {endpoint_name}")

role_arn = os.getenv('SP_ROLE_ARN', 'arn:aws:iam::719805443631:role/bi_sagemaker')
print(f"Role ARN: {role_arn}")

# XGBoost container image URI (adjust region as needed)
xgboost_image_uri = os.getenv(
    'SP_XGBOOST_IMAGE_URI',
    '683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1',
)
print(f"XGBoost Image URI: {xgboost_image_uri}")

Model Name: oejr-prob-xgb-model-20260226
Endpoint Config Name: oejr-prob-xgb-endpoint-config-20260226
Endpoint Name: oejr-prob-xgb-endpoint-20260226
Role ARN: arn:aws:iam::719805443631:role/bi_sagemaker
XGBoost Image URI: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1


In [6]:
# Create SageMaker model
create_model_response = sagemaker_client.create_model(
    ModelName=model_name,
    PrimaryContainer={
        'Image': xgboost_image_uri,
        'ModelDataUrl': s3_uri(PATHS.pipeline_output_key),
        'Environment': {
            'SAGEMAKER_PROGRAM': 'inference.py',
            'SAGEMAKER_SUBMIT_DIRECTORY': s3_uri(PATHS.code_prefix),
        },
    },
    ExecutionRoleArn=role_arn,
)

print(f"Model created: {create_model_response['ModelArn']}")

Model created: arn:aws:sagemaker:us-east-1:719805443631:model/oejr-prob-xgb-model-20260226


In [5]:
# Create endpoint configuration
sagemaker_client.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            'VariantName': 'Primary',
            'ModelName': model_name,
            'InitialInstanceCount': 1,
            'InstanceType': 'ml.m5.large',
        }
    ],
)

print(f"Endpoint config created: {endpoint_config_name}")

Endpoint config created: oejr-prob-xgb-endpoint-config-20260226


In [7]:
# Create endpoint
endpoint_response = sagemaker_client.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=endpoint_config_name,
)

print(f"Endpoint created: {endpoint_response['EndpointArn']}")
print("Endpoint status: Creating... (this may take 5-10 minutes)")

Endpoint created: arn:aws:sagemaker:us-east-1:719805443631:endpoint/oejr-prob-xgb-endpoint-20260226
Endpoint status: Creating... (this may take 5-10 minutes)
